In [197]:
import pandas as pd
import numpy as np
import seaborn as sns

from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge


In [198]:
base = ['year', 'engine_hp', 'engine_cylinders', 'number_of_doors', 'highway_mpg', 'popularity', 'city_mpg']

In [199]:
df = pd.read_csv('ds/data.csv')
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [200]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
string_columns = list(df.dtypes[df.dtypes == 'object'].index)
for col in string_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head()

,make,model,year,engine_fuel_type,engine_hp,engine_cylinders,transmission_type,driven_wheels,number_of_doors,market_category,vehicle_size,vehicle_style,highway_mpg,city_mpg,popularity,msrp
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [201]:
df.isnull().sum()

make                    0
model                   0
year                    0
engine_fuel_type        3
engine_hp              69
engine_cylinders       30
transmission_type       0
driven_wheels           0
number_of_doors         6
market_category      3742
vehicle_size            0
vehicle_style           0
highway_mpg             0
city_mpg                0
popularity              0
msrp                    0
dtype: int64

In [202]:
categorical = ['make', 'engine_fuel_type', 'driven_wheels', 'vehicle_style']
df_dummies = pd.get_dummies(df[categorical])
pd.concat([df[base], df_dummies], axis=1)

,year,engine_hp,engine_cylinders,number_of_doors,highway_mpg,popularity,city_mpg,make_Acura,make_Alfa Romeo,make_Aston Martin,...,vehicle_style_Convertible,vehicle_style_Convertible SUV,vehicle_style_Coupe,vehicle_style_Crew Cab Pickup,vehicle_style_Extended Cab Pickup,vehicle_style_Passenger Minivan,vehicle_style_Passenger Van,vehicle_style_Regular Cab Pickup,vehicle_style_Sedan,vehicle_style_Wagon
0,2011,335.0,6.0,2.0,26,3916,19,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,2011,300.0,6.0,2.0,28,3916,19,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,2011,300.0,6.0,2.0,28,3916,20,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,2011,230.0,6.0,2.0,28,3916,18,False,False,False,...,False,False,True,False,False,False,False,False,False,False
4,2011,230.0,6.0,2.0,28,3916,18,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11909,2012,300.0,6.0,4.0,23,204,16,True,False,False,...,False,False,False,False,False,False,False,False,False,False
11910,2012,300.0,6.0,4.0,23,204,16,True,False,False,...,False,False,False,False,False,False,False,False,False,False
11911,2012,300.0,6.0,4.0,23,204,16,True,False,False,...,False,False,False,False,False,False,False,False,False,False
11912,2013,300.0,6.0,4.0,23,204,16,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [203]:
df['age'] = df.year - 2017


In [204]:
df = df.fillna(df[base].mean())
df.isnull().sum()

make                    0
model                   0
year                    0
engine_fuel_type        3
engine_hp               0
engine_cylinders        0
transmission_type       0
driven_wheels           0
number_of_doors         0
market_category      3742
vehicle_size            0
vehicle_style           0
highway_mpg             0
city_mpg                0
popularity              0
msrp                    0
age                     0
dtype: int64

In [205]:
n = len(df)
print(n)

11914


In [206]:
n_train = int(0.6 * n)
n_val = int(0.2 * n)
n_test = int(0.2 * n)

np.random.seed(10)
idx = np.arange(n)
np.random.shuffle(idx)
df_shuffled = df.iloc[idx]

y_train = np.log1p(df_shuffled[:n_train].msrp.values)
y_val = np.log1p(df_shuffled[n_train:n_train + n_val].msrp.values)
y_test = np.log1p(df_shuffled[n_train + n_val:].msrp.values)

# X = df_shuffled[base].values
X_train = df_shuffled[:n_train][base].values
X_val = df_shuffled[n_train:n_train + n_val][base].values
X_test = df_shuffled[n_train + n_val:][base].values

In [207]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [208]:
y_pred = model.predict(X_val)

In [209]:
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [210]:
for alpha in [0, 0.001, 0.01, 0.1, 1, 10]:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    print(alpha, rmse(y_val, y_pred))

0 0.5161308854286877
0.001 0.5161308857041347
0.01 0.5161308881831476
0.1 0.5161309129730107
1 0.5161311608468406
10 0.5161336370850333


In [211]:
for alpha in [0, 0.001, 0.01, 0.1, 1, 10]:
    model = Ridge(alpha=alpha)
    model.fit(X_test, y_test)
    y_pred = model.predict(X_test)
    print(alpha, ' : ', rmse(y_test, y_pred))

0  :  0.5124158699963801
0.001  :  0.5124158699963828
0.01  :  0.5124158699964756
0.1  :  0.5124158700058987
1  :  0.5124158709472441
10  :  0.512415964110604


In [212]:
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])
model = Ridge(10)
model.fit(X_train_val, y_train_val)
y_pred = model.predict(X_train_val)
print(rmse(y_train_val, y_pred))

0.5088313224655381


In [213]:
# Тест
# real price = 31120

ad = {
    'city_mpg': 18,
    'driven_wheels': 'all_wheel_drive',
    'engine_cylinders': 6.0,
    'engine_fuel_type': 'regular_unleaded',
    'engine_hp': 268.0,
    'highway_mpg': 25,
    'make': 'toyota',
    'market_category': 'crossover,performance',
    'model': 'venza',
    'number_of_doors': 4.0,
    'popularity': 2031,
    'transmission_type': 'automatic',
    'vehicle_size': 'midsize',
    'vehicle_style': 'wagon',
    'year': 2013
}

In [214]:
df = pd.DataFrame([ad])
ad_X = df[base].values
y_pred = np.expm1(model.predict(ad_X))
print('Price: ', round(y_pred[0], 2))

Price:  33408.45
